# Importación de librerias necesarias

In [1]:
import pandas as pd

# Carga de datos
El primer paso será guardar los datos en un dataframe. 

In [ ]:
df_aficionados = pd.read_csv('/workspace/aficionados.csv')
df_eventos = pd.read_csv('/workspace/eventos.csv')
df_promociones = pd.read_csv('/workspace/promociones.csv')

# Limpieza de datos
Procedemos a eliminar las filas con valores nulos

In [3]:
df_aficionados = df_aficionados.dropna()
df_eventos = df_eventos.dropna()
df_promociones = df_promociones.dropna()

# Comnbinación de DataFrames
Para poder analizar los datos en conjunto vamos a combinarlos. 

In [4]:
df_merged = df_eventos.merge(df_promociones, on='id_evento').merge(df_aficionados, on='id_evento')
df_merged.head()

,id_evento,nombre_evento,fecha_evento,ubicación_evento,asistentes_totales,id_promocion,medio_publicitario,presupuesto,fecha_inicio,fecha_fin,id_aficionado,nombre_aficionado,edad,género,ubicación_residencia
0,3,Evento 3,2023-01-03,Ciudad A,254,3,Internet,6804,2023-01-11,2023-01-16,2,Aficionado 2,20,F,Ciudad D
1,3,Evento 3,2023-01-03,Ciudad A,254,3,Internet,6804,2023-01-11,2023-01-16,39,Aficionado 39,23,M,Ciudad D
2,3,Evento 3,2023-01-03,Ciudad A,254,3,Internet,6804,2023-01-11,2023-01-16,47,Aficionado 47,48,M,Ciudad D
3,5,Evento 5,2023-01-05,Ciudad C,7647,7,Radio,15109,2023-01-31,2023-02-05,16,Aficionado 16,51,F,Ciudad D
4,5,Evento 5,2023-01-05,Ciudad C,7647,7,Radio,15109,2023-01-31,2023-02-05,19,Aficionado 19,42,M,Ciudad D


# Preguntas de negocio
- ¿Cuál es el evento con mayor asistencia total?

In [5]:
asistencia_maxima = df_eventos['asistentes_totales'].max()
evento_mas_asistido = df_eventos[df_eventos['asistentes_totales'] == asistencia_maxima]

print(f"Asistencia máxima: {asistencia_maxima}")
evento_mas_asistido[['nombre_evento', 'asistentes_totales']]

Asistencia máxima: 9879


,nombre_evento,asistentes_totales
28,Evento 29,9879


Tras el análisis el resultado demuestra que el Evento 29 fue el evento con mayor asistencia total.

- ¿Qué rango de edad asiste más frecuentemente a los eventos?

In [6]:
bins = [0, 18, 30, 45, 60, 100]
labels = ['<18', '18-30', '31-45', '46-60', '60+']

# Creamos la columna rango_edad en el DataFrame combinado
df_merged['rango_edad'] = pd.cut(df_merged['edad'], bins=bins, labels=labels)

# Contamos la frecuencia por rango
conteo_rangos = df_merged['rango_edad'].value_counts()
print(conteo_rangos)
print(f"\nEl rango de edad más frecuente es: {conteo_rangos.idxmax()}")

rango_edad
18-30    15
31-45    15
46-60     9
<18       0
60+       0
Name: count, dtype: int64

El rango de edad más frecuente es: 18-30


El rango de edad que asiste más frecuentemente a los eventos es entre 18 y 30 años.

- ¿Qué medio publicitario ha generado mayor impacto en términos de asistencia?

In [7]:
df_merged['impacto_promocion'] = df_merged['asistentes_totales'] / df_merged['presupuesto']

# Agrupamos por medio_publicitario para ver la media del impacto
impacto_por_medio = df_merged.groupby('medio_publicitario')['impacto_promocion'].mean().sort_values(ascending=False)

print(impacto_por_medio)
print(f"\nEl medio con mayor impacto es: {impacto_por_medio.idxmax()}")

medio_publicitario
TV                0.504766
Radio             0.491687
Internet          0.451231
Redes Sociales    0.415709
Name: impacto_promocion, dtype: float64

El medio con mayor impacto es: TV


El medio publicitario que ha generado mayor impacto en cuanto a asistencia es la TV. 

- ¿Cuál es la ubicación de residencia que más aficionados aporta a los eventos?

In [8]:
aficionados_por_ubicacion = df_merged.groupby('ubicación_residencia')['id_aficionado'].count().sort_values(ascending=False)

print(aficionados_por_ubicacion)
print(f"\nLa ubicación que más aficionados aporta es: {aficionados_por_ubicacion.idxmax()}")

ubicación_residencia
Ciudad A    13
Ciudad D    12
Ciudad C     8
Ciudad B     6
Name: id_aficionado, dtype: int64

La ubicación que más aficionados aporta es: Ciudad A


Por último, la ubicación de residencia que más aficionados aporta a los eventos es Ciudad A.

# Exportación de resultados
Como último paso vamos a exportar el DataFrame final con todas las transformaciones. 

In [ ]:
df_merged.to_csv('/workspace/reporte_eventos.csv', index=False)